# Extract, modify, and write attribute data in vector datasets

Vector datasets combine geometries with attribute data stored in a table-like structure.
This example shows how to extract, insert, and modify attribute data in vector datasets to prepare them for further analysis. For instance, it calculates the area of all federal states in Germany and adds a random levelized cost of electricity to each state.

**Import required packages**

GeoKit is imported to provide the required functionality for working with vector data.


In [ ]:
import geokit.core.vector
import geokit.core.geom
import geokit.core.srs
import numpy as np
import pandas as pd
import pathlib
from geokit.core.get_test_data import get_test_data

In [ ]:
# Path to cache folder
data_cache_folder = pathlib.Path().cwd().parent.parent.joinpath("geokit", "data")

# Path to shapefile folder
path_to_shape_file = get_test_data(
    file_name="gadm36_DEU_1.shp",
    data_cache_folder=data_cache_folder,
)

# Load vector dataset from a Shapefile as a pandas DataFrame and reproject to EPSG:3035
data_frame_germany: pd.DataFrame = geokit.core.vector.extractFeatures(path_to_shape_file, srs=3035)

**Add attributes to the vector dataset**

In [ ]:
# Add average LCOE column with random values

# Generate random LCOE values for each row
n_rows = data_frame_germany.loc[:, "geom"].shape[0]
lcoe_values = np.random.uniform(low=30.0, high=80.0, size=n_rows)

In [ ]:
# Calculate the size of each geometry (area)
areas_m2 = data_frame_germany.loc[:, "geom"].apply(lambda g: g.GetArea())

# Add new column (LCOE and area)to the vector dataset
data_frame_germany.loc[:, "avg_LCOE"] = pd.Series(lcoe_values)
data_frame_germany.loc[:, "area_km2"] = pd.Series(areas_m2 / 1e6)

# Show updated dataset
data_frame_germany

**Inspect attributes**

In [ ]:
data_frame_germany[["avg_LCOE", "area_km2"]].describe()

**Filter attributes**

In [ ]:
# Select regions with low average LCOE
low_cost_regions = data_frame_germany[data_frame_germany["avg_LCOE"] < 50]

# Show selected regions
print("Regions with low average LCOE (< 50):")
print(low_cost_regions["NAME_1"])


# Select regions higher than average area size
average_area = data_frame_germany["area_km2"].mean()

large_regions = data_frame_germany[data_frame_germany["area_km2"] > average_area]

# Show selected regions
print("")
print("Regions with area larger than average:")
print(large_regions["NAME_1"])

In [ ]:
# Add average LCOE column with random values

# Generate random LCOE values for each row
n_rows = data_frame_germany.geom.shape[0]
lcoe_values = np.random.uniform(low=30.0, high=80.0, size=n_rows)

# Calculate the size of each geometry (area)
areas_m2 = data_frame_germany.geom.apply(lambda g: g.GetArea())

# Add new column (LCOE and area)to the vector dataset
data_frame_germany["avg_LCOE"] = pd.Series(lcoe_values)
data_frame_germany["area_km2"] = pd.Series(areas_m2 / 1e6)

# Show updated dataset
data_frame_germany

### Visualizing vector attributes

In [ ]:
# colored by LCOE
import matplotlib.pyplot as plt

# Create two subplots for side-by-side comparison
fig, ax = plt.subplots(ncols=2, figsize=(10, 5))
ax_handle_lcoe = geokit.core.geom.drawGeoms(
    data_frame_germany,
    colorBy="avg_LCOE",
    srs=3035,
    figsize=(6, 4),
    draw_cbar=True,
    cbarTitle="Average LCOE [€/MWh]",
    ax=ax[0],
)

# colored by area Size

ax_handle_area = geokit.core.geom.drawGeoms(
    data_frame_germany,
    colorBy="area_km2",
    srs=3035,
    figsize=(6, 4),
    draw_cbar=True,
    cbarTitle="Area Size [km²]",
    ax=ax[1],
)

## Store Vector File

Lastly, you can save the new vector file as a shapefile.

In [ ]:
# Simple vector creation
geokit.core.vector.createVector(geoms=data_frame_germany, output="updated_vector_file.shp")